# Starter: Element Cytos A549 CRISPR screen (cell paint + protein + guide + 3' RNA, same cells)

Uses the **tables** tier (1.5 GB). The image section at the end uses the **subset_A1_A2** tier. See the README for the design and the meaning of each column.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd

DATA = Path("data")  # folder holding tables.zarr (and optionally subset_A1_A2.zarr)
sdata = sd.read_zarr(DATA / "tables.zarr")
sdata

## The cell tables

The four cell tables share the same cells, in the same order, and have identical `obs`.

In [ ]:
rna = sdata.tables["global-table-ThreePrimeUntargeted"]
img = sdata.tables["global-table-imaging"]
guides = sdata.tables["global-table-B01-Cellecta"]
assert (rna.obs_names == img.obs_names).all()
rna.obs[["Well", "condition", "guide", "target_gene", "guide_confidence", "rna_total_counts", "pass_qc"]].head()

In [ ]:
pd.crosstab(rna.obs.target_gene, [rna.obs.condition, rna.obs.pass_qc])

## QC

Nothing has been removed. Filter however you like. `pass_qc` combines all `qc_*` flags.

In [ ]:
keep = rna.obs.pass_qc.values
rna, img = rna[keep].copy(), img[keep].copy()
rna.obs.filter(like="qc_").mean()  # all False now; shown for reference

## RNA: normalise, embed (20k-cell subsample for speed)

In [ ]:
rng = np.random.default_rng(0)
sub = rng.choice(rna.n_obs, 20_000, replace=False)
r = rna[sub].copy()
sc.pp.normalize_total(r, target_sum=1e4)
sc.pp.log1p(r)
sc.pp.highly_variable_genes(r, n_top_genes=2000)
sc.pp.pca(r, mask_var="highly_variable")
sc.pp.neighbors(r)
sc.tl.umap(r)
sc.pl.umap(r, color=["condition", "target_gene", "Well", "rna_total_counts"], ncols=2)

## Imaging features: cell paint + protein + morphology

Drop background channels (`Bkg_*`) and the NaN-prone `*_NuclearRatio` features, then z-score per well.

In [ ]:
feat = ~img.var_names.str.startswith("Bkg_") & ~img.var_names.str.endswith("_NuclearRatio")
m = img[sub][:, feat].copy()
m.X = np.nan_to_num(m.X)
for w in m.obs.Well.unique():  # per-well z-score removes plate/well effects
    i = (m.obs.Well == w).values
    m.X[i] = (m.X[i] - m.X[i].mean(0)) / (m.X[i].std(0) + 1e-6)
sc.pp.pca(m)
sc.pp.neighbors(m)
sc.tl.umap(m)
sc.pl.umap(m, color=["condition", "target_gene", "Phospho-p38", "Nucleus"], ncols=2)

## One perturbation, three modalities: TGFBR2 KO vs non-targeting

In [ ]:
def pseudobulk(adata, genes):
    a = adata[:, genes]
    x = pd.DataFrame(a.X.toarray() if hasattr(a.X, "toarray") else a.X, columns=genes, index=a.obs_names)
    return x.groupby([a.obs.Well, a.obs.target_gene], observed=True).mean()

rn = rna.copy()
sc.pp.normalize_total(rn, target_sum=1e4)
sc.pp.log1p(rn)
genes = ["TGFBR2", "PMEPA1", "TGFBI", "CDH2", "IGFBP7"]
pb = pseudobulk(rn, genes)
lfc = pb.xs("TGFBR2", level="target_gene") - pb.xs("NTC", level="target_gene")
lfc.plot.box(title="TGFBR2 KO vs NTC, per well (log-normalised RNA)"); plt.axhline(0, c="k", lw=0.5);

In [ ]:
geom = img.obsm["cell_geometry"].join(img.obs[["Well", "target_gene"]])
g = geom.groupby(["Well", "target_gene"], observed=True)[["AreaUm", "NuclearAreaUm", "AreaShape_Eccentricity", "AreaShape_FormFactor"]].median()
(g.xs("TGFBR2", level="target_gene") / g.xs("NTC", level="target_gene")).plot.box(title="TGFBR2 KO / NTC morphology, per well"); plt.axhline(1, c="k", lw=0.5);

## Images (subset tier)

Crop a 1,000 px tile from well A1 at full resolution and overlay the cell masks.

In [ ]:
if (DATA / "subset_A1_A2.zarr").exists():
    s2 = sd.read_zarr(DATA / "subset_A1_A2.zarr")
    im = sd.get_pyramid_levels(s2.images["A1-images-cell_paint"], n=0)
    lab = sd.get_pyramid_levels(s2.labels["A1-labels-Cell"], n=0)
    y, x = slice(6000, 7000), slice(6000, 7000)
    chans = ["Nucleus", "Actin", "Mitochondria", "Cell-Membrane", "Phospho-p38"]
    fig, ax = plt.subplots(1, len(chans) + 1, figsize=(4 * (len(chans) + 1), 4))
    for a, c in zip(ax, chans):
        a.imshow(im.sel(c=c)[y, x].values, cmap="gray", vmax=np.percentile(im.sel(c=c)[y, x].values, 99.5)); a.set_title(c); a.axis("off")
    m = np.asarray(lab[y, x]); ax[-1].imshow(np.ma.masked_equal(m, 0) % 20, cmap="tab20", interpolation="nearest"); ax[-1].set_title("cell masks"); ax[-1].axis("off")